# EDA LMS Цифриум — структурированная версия

Новый ноутбук повторяет рабочие шаги из `EDA.ipynb`, но оформляет их в линейный пайплайн для дальнейших экспериментов. Проверка кириллицы: кириллица сохраняется корректно.

**Структура анализа**

1. Этап подготовки — чтение данных, аудит схем и очистка признаков.
2. Этап слияния — агрегации до уровня `user-course` и контроль merge-предположений.
3. Этап анализа — описательная статистика и гипотезы по оттоку.

## Этап 1. Подготовка и чистка таблиц

### 1.1 Импорты и глобальные настройки

In [ ]:

import pandas as pd
import numpy as np
import json
from pathlib import Path
from IPython.display import display
import warnings

warnings.filterwarnings('ignore')

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)

NOTEBOOK_DIR = Path.cwd().resolve()
POSSIBLE_ROOTS = [
    NOTEBOOK_DIR,
    NOTEBOOK_DIR.parent,
    NOTEBOOK_DIR.parent.parent,
]

DATA_DIR = None
DATA_DIR_CANDIDATES = []
for root in POSSIBLE_ROOTS:
    candidate = root / 'data' / 'raw'
    DATA_DIR_CANDIDATES.append(candidate)
    if candidate.exists():
        PROJECT_ROOT = root
        DATA_DIR = candidate
        break

if DATA_DIR is None:
    raise FileNotFoundError('Не удалось найти каталог data/raw рядом с ноутбуком. Проверьте структуру Xakaton/data/raw.')

print(f'Корень проекта: {PROJECT_ROOT}')
print(f'Каталог с исходными CSV: {DATA_DIR}')


### 1.2 Файлы и колонки с датами

In [ ]:

FILES = {
    'users_courses': 'users_courses.csv',
    'user_answers': 'user_answers.csv',
    'user_activity_histories': 'user_activity_histories.csv',
    'user_trainings': 'user_trainings.csv',
    'wk_users_courses_actions': 'wk_users_courses_actions.csv',
    'lessons': 'lessons.csv',
    'user_lessons': 'user_lessons.csv',
    'wk_media_view_sessions': 'wk_media_view_sessions.csv',
    'user_access_histories': 'user_access_histories.csv',
    'user_award_badges': 'user_award_badges.csv',
    'users': 'users.csv'
}

DATE_COLS = {
    'users_courses': ['access_finished_at', 'wk_officially_started_at', 'wk_course_completed_at', 'created_at', 'updated_at'],
    'user_answers': ['submitted_at'],
    'user_activity_histories': ['created_at'],
    'user_trainings': ['started_at', 'finished_at', 'mark_saved_at'],
    'wk_users_courses_actions': ['created_at', 'updated_at'],
    'wk_media_view_sessions': ['started_at'],
    'user_access_histories': ['access_started_at', 'access_expired_at'],
    'user_award_badges': ['created_at'],
    'users': ['created_at', 'grade_changed_at', 'updated_at', 'd_updated_at', 'remember_created_at', 'current_sign_in_at', 'last_sign_in_at'],
    'lessons': ['wk_attendance_tracking_disabled_at']
}


### 1.3 Загрузка и первичный аудит

In [ ]:

def load_all_tables(data_dir: Path, files: dict, date_cols: dict):
    dfs = {}
    overview = []
    for name, filename in files.items():
        path = data_dir / filename
        if not path.exists():
            print(f'!!! Не найден файл: {path}')
            continue
        parse_dates = date_cols.get(name, [])
        df = pd.read_csv(path, index_col=0, parse_dates=parse_dates, low_memory=False)
        dfs[name] = df
        overview.append({
            'table': name,
            'rows': len(df),
            'cols': df.shape[1],
            'memory_mb': round(df.memory_usage(deep=True).sum() / (1024**2), 2),
            'parsed_dates': len(parse_dates)
        })
    overview_df = pd.DataFrame(overview).sort_values('rows', ascending=False).reset_index(drop=True)
    return dfs, overview_df

dfs, tables_overview = load_all_tables(DATA_DIR, FILES, DATE_COLS)
tables_overview


### 1.4 Полные дубликаты строк

In [ ]:

def summarize_duplicates(dfs: dict) -> pd.DataFrame:
    rows = []
    for name, df in dfs.items():
        dup_count = df.duplicated().sum()
        rows.append({
            'table': name,
            'rows': len(df),
            'duplicates': dup_count,
            'duplicates_%': round((dup_count / len(df)) * 100, 2) if len(df) else 0.0
        })
    summary = pd.DataFrame(rows).sort_values('duplicates_%', ascending=False)
    return summary

duplicate_report = summarize_duplicates(dfs)
duplicate_report


### 1.5 Приведение целочисленных колонок и master-ключей

In [ ]:

INT_COLS_MAP = {
    'users_courses': ['user_id', 'course_id'],
    'user_answers': ['user_id', 'task_id'],
    'user_activity_histories': ['user_lesson_id'],
    'user_trainings': ['user_id', 'training_id'],
    'wk_users_courses_actions': ['user_id', 'users_course_id', 'lesson_id'],
    'lessons': ['course_id', 'lesson_number', 'wk_task_count'],
    'user_lessons': ['user_id', 'lesson_id', 'group_id', 'users_course_id'],
    'user_award_badges': ['user_id', 'award_badge_id'],
    'wk_media_view_sessions': ['viewer_id', 'segments_total', 'viewed_segments_count'],
    'users': ['id', 'sign_in_count', 'grade_id', 'd_wk_school_id', 'd_wk_municipal_id', 'd_wk_region_id']
}

def convert_int_like_columns(dfs: dict, int_cols_map: dict):
    for table, cols in int_cols_map.items():
        if table not in dfs:
            continue
        for col in cols:
            if col not in dfs[table].columns:
                continue
            if pd.api.types.is_integer_dtype(dfs[table][col]):
                continue
            dfs[table][col] = pd.to_numeric(
                dfs[table][col].astype(str).str.replace(',', ', regex=False),
                errors='coerce'
            ).astype('Int64')

convert_int_like_columns(dfs, INT_COLS_MAP)

if 'users_courses' in dfs:
    dfs['users_courses'] = dfs['users_courses'].rename_axis('users_course_id').reset_index()


### 1.6 Точечная очистка таблиц

In [ ]:

def preprocess_users_courses(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['is_active'] = df['state'].str.lower().eq('active')
    for col in ['wk_max_points', 'wk_max_task_count']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['progress_pct'] = np.where(
        (df['wk_max_points'] > 0) & (df['wk_max_points'].notna()),
        (df['wk_points'] / df['wk_max_points'] * 100).round(2),
        np.nan
    )
    return df.drop(columns=['state', 'wk_points'])

def preprocess_user_answers(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in ['solved', 'skipped', 'wk_partial_answer']:
        df[col] = df[col].astype(str).str.strip().str.lower().eq('true')
    df['attempts_pct'] = np.where(
        (df['max_attempts'] > 0) & (df['max_attempts'].notna()),
        (df['attempts'] / df['max_attempts'] * 100).round(2),
        np.nan
    )
    df['is_lesson'] = df['resource_type'].eq('Lesson')
    df['is_training'] = df['resource_type'].eq('Training')
    df['is_hw'] = df['resource_type'].eq('Homework')

    def parse_results(value):
        if isinstance(value, str):
            try:
                return json.loads(value)
            except json.JSONDecodeError:
                return []
        return []

    df['parsed_results'] = df['results'].apply(parse_results)
    df = df.explode('parsed_results').reset_index(drop=True)

    def extract_dict(d):
        if not isinstance(d, dict) or not d:
            return (None, None, None)
        key = next(iter(d.keys()))
        payload = next(iter(d.values()))
        return (
            pd.to_numeric(key, errors='coerce'),
            payload.get('points'),
            payload.get('coefficient')
        )

    extracted = df['parsed_results'].apply(extract_dict)
    df['result_subtask_id'] = extracted.apply(lambda x: x[0]).astype('Int64')
    df['result_points'] = pd.to_numeric(extracted.apply(lambda x: x[1]), errors='coerce')
    df['result_coefficient'] = pd.to_numeric(extracted.apply(lambda x: x[2]), errors='coerce')

    return df.drop(columns=['parsed_results', 'results', 'performance', 'attempts', 'resource_type'])

def preprocess_user_activity_histories(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['is_visit_video'] = df['action'].eq('visit_video')
    df['is_visit_translation'] = df['action'].eq('visit_translation')
    df['is_show_conspect'] = df['action'].eq('show_conspect')
    return df.drop(columns=['action'])

def preprocess_user_trainings(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['is_lesson_training'] = df['type'].eq('UserTrainings::LessonTraining')
    df['is_regular_training'] = df['type'].eq('UserTrainings::RegularTraining')
    df['is_olympiad_training'] = df['type'].eq('UserTrainings::OlympiadTraining')
    df['is_started'] = df['state'].eq('started')
    return df.drop(columns=['type', 'state'])

def preprocess_wk_users_courses_actions(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['is_visit_video'] = df['action'].eq('visit_video')
    df['is_show_conspect'] = df['action'].eq('visit_preparation_material')
    df['is_start_training'] = df['action'].eq('start_training')
    df['is_user_answer'] = df['action'].eq('user_answer')
    df['is_visit_translation'] = df['action'].eq('visit_translation')
    df['is_scratch_playground_visited'] = df['action'].eq('scratch_playground_visited')
    return df.drop(columns=['action', 'sourceable_id', 'lesson_id'])

def preprocess_user_award_badges(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    pivot = df.pivot_table(index='user_id', columns='award_badge_id', values='created_at', aggfunc='min')
    date_cols = {f'date_badge_{int(c)}': pivot[c] for c in pivot.columns}
    dates = pd.DataFrame(date_cols)
    flag_cols = {f'has_badge_{int(c)}': pivot[c].notna() for c in pivot.columns}
    flags = pd.DataFrame(flag_cols)
    return pd.concat([flags, dates], axis=1).reset_index()

def preprocess_users(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['user_id'] = df['id']
    drop_cols = ['is_teacher', 'current_sign_in_at', 'last_sign_in_at', 'grade_checked', 'xp', 'd_updated_at']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    df['is_teacher'] = df['type'].str.contains('Agent', na=False)
    df = df.drop(columns=['type'])
    return df[df['is_teacher'] == False].reset_index(drop=True)

def preprocess_user_access_histories(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['is_premium_access'] = df['activator_class'].eq('Courses::AccessActivators::PremiumAccessActivator')
    df['is_revoke_access'] = df['activator_class'].eq('Courses::AccessActivators::RevokeAccessActivator')
    df['is_standard_access'] = df['activator_class'].eq('Courses::AccessActivators::StandardAccessActivator')
    df['is_change_duration_access'] = df['activator_class'].eq('Courses::AccessActivators::ChangeAccessDurationActivator')
    df['is_month_premium_access'] = df['activator_class'].eq('Courses::AccessActivators::MonthPremiumAccessActivator')
    return df.drop(columns=['activator_class'])

def preprocess_user_lessons(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in ['video_visited', 'translation_visited', 'solved', 'video_viewed']:
        if col in df.columns:
            df[col] = df[col].fillna(False).astype(bool)
    return df

def preprocess_lessons(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    bool_cols = ['conspect_expected', 'task_expected', 'wk_survival_training_expected',
                 'wk_scratch_playground_enabled', 'wk_attendance_tracking_enabled']
    for col in bool_cols:
        if col in df.columns:
            df[col] = df[col].fillna(False).astype(bool)
    df['wk_video_duration'] = pd.to_numeric(df['wk_video_duration'], errors='coerce')
    return df

def preprocess_wk_media_view_sessions(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['segments_total'] = pd.to_numeric(df['segments_total'], errors='coerce')
    df['viewed_segments_count'] = pd.to_numeric(df['viewed_segments_count'], errors='coerce')
    return df

TABLE_PREPROCESSORS = {
    'users_courses': preprocess_users_courses,
    'user_answers': preprocess_user_answers,
    'user_activity_histories': preprocess_user_activity_histories,
    'user_trainings': preprocess_user_trainings,
    'wk_users_courses_actions': preprocess_wk_users_courses_actions,
    'user_award_badges': preprocess_user_award_badges,
    'users': preprocess_users,
    'user_access_histories': preprocess_user_access_histories,
    'user_lessons': preprocess_user_lessons,
    'lessons': preprocess_lessons,
    'wk_media_view_sessions': preprocess_wk_media_view_sessions
}

for name, func in TABLE_PREPROCESSORS.items():
    if name in dfs:
        dfs[name] = func(dfs[name])


### 1.7 Быстрый дата-качественный отчёт по ключевым таблицам

In [ ]:

def eda_report(dfs_subset: dict, max_cat_examples: int = 3):
    with pd.option_context('display.max_columns', None, 'display.width', 120):
        for name, df in dfs_subset.items():
            print(f"
{'='*20} {name} {'='*20}")
            mem_mb = df.memory_usage(deep=True).sum() / (1024**2)
            missing_total = df.isnull().sum().sum()
            cell_count = df.shape[0] * max(df.shape[1], 1)
            missing_pct = (missing_total / cell_count * 100) if cell_count else 0
            print(f"Форма: {df.shape[0]:,} x {df.shape[1]} | Память: {mem_mb:.2f} MB | Пропуски: {missing_total:,} ({missing_pct:.2f}%)")

            num_cols = df.select_dtypes(include=['number']).columns
            if len(num_cols) > 0:
                num_stats = df[num_cols].describe(percentiles=[0.5]).T[['count', 'mean', 'std', 'min', '50%', 'max']]
                num_stats['missing_%'] = (df[num_cols].isnull().mean() * 100).round(0)
                print('Числовые признаки:')
                print(num_stats.to_string())

            bool_cols = df.select_dtypes(include='bool').columns
            if len(bool_cols) > 0:
                rows = []
                for col in bool_cols:
                    vc = df[col].value_counts(normalize=True)
                    rows.append({
                        'признак': col,
                        'true_%': round(vc.get(True, 0) * 100, 1),
                        'false_%': round(vc.get(False, 0) * 100, 1)
                    })
                print('Булевые признаки:')
                print(pd.DataFrame(rows).to_string(index=False))

            cat_cols = df.select_dtypes(include=['object']).columns
            if len(cat_cols) > 0:
                rows = []
                for col in cat_cols:
                    vc = df[col].value_counts(normalize=True).head(max_cat_examples)
                    rows.append({
                        'признак': col,
                        'уникальных': df[col].nunique(dropna=True),
                        'топ-значения': ' | '.join([f"{idx} ({val:.0%})" for idx, val in vc.items()])
                    })
                print('Категориальные признаки:')
                print(pd.DataFrame(rows).to_string(index=False))

key_tables = ['users_courses', 'user_answers', 'wk_users_courses_actions', 'user_lessons']
eda_report({name: dfs[name] for name in key_tables if name in dfs})


## Этап 2. Слияние таблиц и подготовка фичей

На этом этапе все источники приводим к уровню `user-course`, проговариваем предположения по ключам и фиксируем новые признаки.

### 2.1 Функции для агрегирования

In [ ]:

def build_course_features_from_lessons(lessons: pd.DataFrame) -> pd.DataFrame:
    df = lessons.copy()
    if df.empty:
        return pd.DataFrame(columns=['course_id'])

    numeric_cols = ['lesson_number', 'wk_max_points', 'wk_task_count', 'wk_video_duration']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    bool_cols = ['conspect_expected', 'task_expected', 'wk_survival_training_expected',
                 'wk_scratch_playground_enabled', 'wk_attendance_tracking_enabled']
    for col in bool_cols:
        df[col] = df[col].fillna(False).astype(bool)

    df['has_lesson_number'] = df['lesson_number'].notna()
    df['has_tasks_observed'] = df['wk_task_count'].fillna(0).gt(0) | df['wk_max_points'].fillna(0).gt(0)
    df['has_video'] = df['wk_video_duration'].fillna(0).gt(0)
    df['attendance_disabled_marked'] = df['wk_attendance_tracking_disabled_at'].notna()
    df['task_flag_mismatch'] = (~df['task_expected']) & df['has_tasks_observed']

    valid_task_rows = df['wk_task_count'].fillna(0).gt(0) & df['wk_max_points'].notna()
    df['points_per_task'] = np.where(valid_task_rows, df['wk_max_points'] / df['wk_task_count'], np.nan)
    df['points_equal_task'] = np.where(valid_task_rows, np.isclose(df['wk_max_points'], df['wk_task_count'], atol=1e-9), np.nan)

    course_features = (
        df.groupby('course_id', dropna=False)
        .agg(
            lessons_total_count=('course_id', 'size'),
            lessons_with_number_count=('has_lesson_number', 'sum'),
            lessons_with_conspect_count=('conspect_expected', 'sum'),
            lessons_task_expected_rows=('task_expected', 'sum'),
            lessons_attendance_enabled_rows=('wk_attendance_tracking_enabled', 'sum'),
            lessons_attendance_disabled_marked_rows=('attendance_disabled_marked', 'sum'),
            lessons_survival_rows=('wk_survival_training_expected', 'sum'),
            lessons_scratch_rows=('wk_scratch_playground_enabled', 'sum'),
            lessons_rows_with_tasks_observed=('has_tasks_observed', 'sum'),
            lessons_task_count_sum=('wk_task_count', 'sum'),
            lessons_task_count_mean=('wk_task_count', 'mean'),
            lessons_task_count_max=('wk_task_count', 'max'),
            lessons_max_points_sum=('wk_max_points', 'sum'),
            lessons_max_points_mean=('wk_max_points', 'mean'),
            lessons_max_points_max=('wk_max_points', 'max'),
            lessons_points_per_task_mean=('points_per_task', 'mean'),
            lessons_points_equal_task_share=('points_equal_task', 'mean'),
            lessons_rows_with_video=('has_video', 'sum'),
            lessons_video_duration_sum=('wk_video_duration', 'sum'),
            lessons_video_duration_mean=('wk_video_duration', 'mean'),
            lessons_video_duration_max=('wk_video_duration', 'max'),
            lessons_task_flag_mismatch_rows=('task_flag_mismatch', 'sum'),
        )
        .reset_index()
    )

    denom = course_features['lessons_total_count'].replace(0, np.nan)
    share_map = {
        'lessons_with_number_share': 'lessons_with_number_count',
        'lessons_conspect_expected_share': 'lessons_with_conspect_count',
        'lessons_task_expected_share': 'lessons_task_expected_rows',
        'lessons_tasks_observed_share': 'lessons_rows_with_tasks_observed',
        'lessons_video_share': 'lessons_rows_with_video',
        'lessons_attendance_enabled_share': 'lessons_attendance_enabled_rows',
        'lessons_survival_share': 'lessons_survival_rows',
        'lessons_scratch_share': 'lessons_scratch_rows',
        'lessons_task_flag_mismatch_share': 'lessons_task_flag_mismatch_rows'
    }
    for new_col, base_col in share_map.items():
        course_features[new_col] = (course_features[base_col] / denom).round(3)

    ordered_cols = ['course_id', 'lessons_total_count', 'lessons_task_count_sum', 'lessons_task_count_mean',
                    'lessons_task_count_max', 'lessons_max_points_sum', 'lessons_max_points_mean',
                    'lessons_max_points_max', 'lessons_points_per_task_mean', 'lessons_video_duration_sum',
                    'lessons_video_duration_mean', 'lessons_video_duration_max', 'lessons_points_equal_task_share',
                    'lessons_with_number_share', 'lessons_conspect_expected_share', 'lessons_task_expected_share',
                    'lessons_tasks_observed_share', 'lessons_video_share', 'lessons_attendance_enabled_share',
                    'lessons_survival_share', 'lessons_scratch_share', 'lessons_task_flag_mismatch_share']
    return course_features[ordered_cols].fillna(0)


In [ ]:

def aggregate_user_lessons(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=['users_course_id'])
    agg = df.groupby('users_course_id').agg(
        lessons_touched=('lesson_id', 'nunique'),
        lessons_solved=('solved', 'sum'),
        solved_tasks_total=('wk_solved_task_count', 'sum'),
        lesson_points_total=('wk_points', 'sum'),
        video_visited_share=('video_visited', 'mean'),
        translation_visited_share=('translation_visited', 'mean')
    ).reset_index()
    agg['lessons_solved_share'] = np.where(
        agg['lessons_touched'] > 0,
        (agg['lessons_solved'] / agg['lessons_touched'] * 100).round(1),
        np.nan
    )
    agg['video_visited_share'] = (agg['video_visited_share'] * 100).round(1)
    agg['translation_visited_share'] = (agg['translation_visited_share'] * 100).round(1)
    return agg

def aggregate_course_actions(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=['users_course_id'])
    agg = df.groupby('users_course_id').agg(
        total_actions=('users_course_id', 'size'),
        first_action_at=('created_at', 'min'),
        last_action_at=('created_at', 'max'),
        visit_video_count=('is_visit_video', 'sum'),
        show_conspect_count=('is_show_conspect', 'sum'),
        start_training_count=('is_start_training', 'sum'),
        user_answer_count=('is_user_answer', 'sum'),
        visit_translation_count=('is_visit_translation', 'sum'),
        scratch_playground_count=('is_scratch_playground_visited', 'sum')
    ).reset_index()
    agg['active_days'] = (agg['last_action_at'] - agg['first_action_at']).dt.days
    return agg

def aggregate_user_answers(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=['user_id'])
    agg = df.groupby('user_id').agg(
        answers_total=('task_id', 'count'),
        solved_share=('solved', 'mean'),
        skipped_share=('skipped', 'mean'),
        lesson_answer_share=('is_lesson', 'mean'),
        training_answer_share=('is_training', 'mean'),
        hw_answer_share=('is_hw', 'mean'),
        avg_attempts_pct=('attempts_pct', 'mean'),
        avg_result_points=('result_points', 'mean')
    ).reset_index()
    share_cols = ['solved_share', 'skipped_share', 'lesson_answer_share', 'training_answer_share', 'hw_answer_share']
    for col in share_cols:
        agg[col] = (agg[col] * 100).round(1)
    return agg

def aggregate_user_access_histories(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=['users_course_id'])
    agg = df.groupby('users_course_id').agg(
        access_events=('users_course_id', 'size'),
        first_access_at=('access_started_at', 'min'),
        last_access_at=('access_expired_at', 'max'),
        premium_access_count=('is_premium_access', 'sum'),
        revoke_access_count=('is_revoke_access', 'sum'),
        standard_access_count=('is_standard_access', 'sum'),
        month_premium_count=('is_month_premium_access', 'sum'),
        change_duration_count=('is_change_duration_access', 'sum')
    ).reset_index()
    agg['access_duration_days'] = (agg['last_access_at'] - agg['first_access_at']).dt.days
    return agg

def aggregate_user_trainings(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=['user_id'])
    agg = df.groupby('user_id').agg(
        trainings_total=('training_id', 'nunique'),
        trainings_started_share=('is_started', 'mean'),
        lesson_trainings=('is_lesson_training', 'sum'),
        regular_trainings=('is_regular_training', 'sum'),
        olympiad_trainings=('is_olympiad_training', 'sum'),
        total_training_points=('earned_points', 'sum'),
        total_training_attempts=('attempts', 'sum')
    ).reset_index()
    agg['trainings_started_share'] = (agg['trainings_started_share'] * 100).round(1)
    return agg

def enrich_users_with_badges(users_df: pd.DataFrame, badges_df: pd.DataFrame) -> pd.DataFrame:
    users_df = users_df.copy()
    if badges_df is None or badges_df.empty:
        return users_df
    merged = users_df.merge(badges_df, on='user_id', how='left')
    badge_cols = [col for col in merged.columns if col.startswith('has_badge_')]
    for col in badge_cols:
        merged[col] = merged[col].fillna(False)
    return merged


### 2.2 Расчёт агрегатов

In [ ]:

lessons_course_features = build_course_features_from_lessons(dfs.get('lessons', pd.DataFrame()))
user_lessons_features = aggregate_user_lessons(dfs.get('user_lessons', pd.DataFrame()))
course_actions_features = aggregate_course_actions(dfs.get('wk_users_courses_actions', pd.DataFrame()))
user_answers_features = aggregate_user_answers(dfs.get('user_answers', pd.DataFrame()))
user_access_features = aggregate_user_access_histories(dfs.get('user_access_histories', pd.DataFrame()))
user_training_features = aggregate_user_trainings(dfs.get('user_trainings', pd.DataFrame()))
users_enriched = enrich_users_with_badges(dfs.get('users', pd.DataFrame()), dfs.get('user_award_badges', pd.DataFrame()))

feature_shapes = pd.DataFrame({
    'датасет': [
        'lessons_course_features', 'user_lessons_features', 'course_actions_features',
        'user_answers_features', 'user_access_features', 'user_training_features', 'users_enriched'
    ],
    'размер': [
        lessons_course_features.shape,
        user_lessons_features.shape,
        course_actions_features.shape,
        user_answers_features.shape,
        user_access_features.shape,
        user_training_features.shape,
        users_enriched.shape
    ]
})
feature_shapes


### 2.3 Предположения по merge

- `users_courses` — мастер-таблица уровня user-course.
- `lessons_course_features` добавляется many-to-one по `course_id`.
- `user_lessons`, `wk_users_courses_actions` и `user_access_histories` агрегированы по `users_course_id` и жёстко соответствуют конкретному курсу пользователя.
- `user_answers` и `user_trainings` агрегируются по `user_id`, поэтому их признаки размножаются на все курсы пользователя (предположение, требующее отдельной валидации).
- `users_enriched` добавляет профиль пользователя и дату получения бейджей.
- После каждого merge проверяем размер таблицы и фиксируем число новых колонок.

### 2.4 Сборка master-таблицы

In [ ]:

user_course_features = dfs['users_courses'].copy()

merge_steps = [
    {'right': lessons_course_features, 'on': 'course_id', 'validate': 'many_to_one', 'note': 'course_id > метаданные уроков'},
    {'right': user_lessons_features, 'on': 'users_course_id', 'validate': 'one_to_one', 'note': 'user-course > прогресс по урокам'},
    {'right': course_actions_features, 'on': 'users_course_id', 'validate': 'one_to_one', 'note': 'user-course > действия на курсе'},
    {'right': user_access_features, 'on': 'users_course_id', 'validate': 'one_to_one', 'note': 'user-course > история доступов'},
    {'right': users_enriched, 'on': 'user_id', 'validate': 'many_to_one', 'note': 'user_id > профиль и бейджи'},
    {'right': user_answers_features, 'on': 'user_id', 'validate': 'many_to_one', 'note': 'user_id > ответы (assumption)'},
    {'right': user_training_features, 'on': 'user_id', 'validate': 'many_to_one', 'note': 'user_id > тренировки (assumption)'}
]

merge_log = []
for step in merge_steps:
    right = step['right']
    if right is None or right.empty:
        merge_log.append({'шаг': step['note'], 'статус': 'пропущено', 'причина': 'пустой датасет'})
        continue
    before_cols = user_course_features.shape[1]
    user_course_features = user_course_features.merge(
        right,
        on=step['on'],
        how='left',
        validate=step.get('validate', 'many_to_one')
    )
    merge_log.append({
        'шаг': step['note'],
        'статус': 'готово',
        'новых_колонок': user_course_features.shape[1] - before_cols
    })

if {'lessons_max_points_sum', 'lessons_task_count_sum'}.issubset(user_course_features.columns):
    user_course_features = user_course_features.drop(columns=['lessons_max_points_sum', 'lessons_task_count_sum'])

pd.DataFrame(merge_log)


### 2.5 Производные признаки

In [ ]:

user_course_features['lessons_solved_share'] = np.where(
    user_course_features.get('lessons_touched', 0) > 0,
    (user_course_features.get('lessons_solved', 0) / user_course_features.get('lessons_touched', 1) * 100).round(1),
    np.nan
)
user_course_features['inactivity_gap_days'] = (
    user_course_features['access_finished_at'] - user_course_features['last_action_at']
).dt.days
user_course_features['preexpiry_inactivity_flag'] = user_course_features['inactivity_gap_days'].gt(14)
user_course_features['access_to_start_days'] = (
    user_course_features['access_finished_at'] - user_course_features['wk_officially_started_at']
).dt.days
user_course_features['actions_per_day'] = np.where(
    user_course_features['active_days'].gt(0),
    (user_course_features['total_actions'] / user_course_features['active_days']).round(2),
    np.nan
)
user_course_features.head()


## Этап 3. Анализ подготовленных таблиц

### 3.1 Обзор итогового датасета

In [ ]:

final_overview = {
    'строк': user_course_features.shape[0],
    'столбцов': user_course_features.shape[1],
    'объём_MB': round(user_course_features.memory_usage(deep=True).sum() / (1024**2), 2)
}
print(final_overview)
user_course_features.info(verbose=False)


### 3.2 Пропуски

In [ ]:

missing_pct = (user_course_features.isnull().mean() * 100).sort_values(ascending=False)
missing_pct.head(25)


### 3.3 Распределения ключевых метрик

In [ ]:

key_metrics = ['progress_pct', 'lessons_touched', 'lessons_solved_share', 'total_actions', 'active_days', 'access_duration_days', 'inactivity_gap_days']
available_metrics = [col for col in key_metrics if col in user_course_features.columns]
metrics_summary = user_course_features[available_metrics].describe(percentiles=[0.25, 0.5, 0.75, 0.9]).T if available_metrics else pd.DataFrame()
metrics_summary


### 3.4 Активные и неактивные пользователи

In [ ]:

state_cols = ['progress_pct', 'total_actions', 'lessons_solved_share', 'inactivity_gap_days']
available_state_cols = [col for col in state_cols if col in user_course_features.columns]
state_summary = user_course_features.groupby('is_active')[available_state_cols].median() if available_state_cols else pd.DataFrame()
state_summary


### 3.5 Контроль расхождений решённых задач

In [ ]:

if 'user_lessons' in dfs and not dfs['user_lessons'].empty:
    lesson_quality = dfs['user_lessons'].assign(
        solved_diff=lambda d: d['solved_tasks_count'] - d['wk_solved_task_count']
    )
    mismatches = lesson_quality[lesson_quality['solved_diff'].ne(0)]
    print(f"Найдено {len(mismatches):,} строк с расхождением solved_tasks_count vs wk_solved_task_count")
    display(mismatches[['users_course_id', 'lesson_id', 'solved_tasks_count', 'wk_solved_task_count', 'solved_diff']].head())
else:
    print('Нет таблицы user_lessons для проверки.')


### 3.6 Сопоставление суммарных баллов курса

In [ ]:

if 'users_courses' in dfs and not lessons_course_features.empty:
    course_check = dfs['users_courses'][['course_id', 'wk_max_points']].merge(
        lessons_course_features[['course_id', 'lessons_max_points_max']],
        on='course_id',
        how='left'
    )
    course_check['points_diff'] = course_check['wk_max_points'] - course_check['lessons_max_points_max']
    mismatch_courses = course_check[course_check['points_diff'].ne(0)]
    print(f"Курсов с расхождениями: {mismatch_courses['course_id'].nunique()} из {course_check['course_id'].nunique()}")
    display(mismatch_courses.head())

    COURSE_ID_TO_INSPECT = 170000688
    sample_course = course_check[course_check['course_id'] == COURSE_ID_TO_INSPECT]
    if not sample_course.empty:
        lessons_sum = dfs['lessons'][dfs['lessons']['course_id'] == COURSE_ID_TO_INSPECT]['wk_max_points'].sum()
        print(f"Курс {COURSE_ID_TO_INSPECT}: wk_max_points из users_courses vs сумма по lessons = {lessons_sum}")
        display(sample_course)
else:
    print('Недостаточно данных для проверки баллов курса.')


### 3.7 Точечные проверки user-course

In [ ]:

EXAMPLE_USER_ID = 665744
EXAMPLE_COURSE_ID = 50000592
EXAMPLE_USERS_COURSE_ID = 449093
SECOND_USERS_COURSE_ID = 648983
SECOND_USER_ID = 745551

examples = {
    'users_courses — пользователь 745551': dfs.get('users_courses', pd.DataFrame()).query('user_id == @SECOND_USER_ID'),
    'user_lessons — users_course_id 648983': dfs.get('user_lessons', pd.DataFrame()).query('users_course_id == @SECOND_USERS_COURSE_ID'),
    'users_courses — пользователь 665744': dfs.get('users_courses', pd.DataFrame()).query('user_id == @EXAMPLE_USER_ID'),
    'wk_users_courses_actions — users_course_id 449093': dfs.get('wk_users_courses_actions', pd.DataFrame()).query('users_course_id == @EXAMPLE_USERS_COURSE_ID'),
    'user_answers — пользователь 665744': dfs.get('user_answers', pd.DataFrame()).query('user_id == @EXAMPLE_USER_ID'),
    'user_trainings — пользователь 665744': dfs.get('user_trainings', pd.DataFrame()).query('user_id == @EXAMPLE_USER_ID'),
    'user_access_histories — users_course_id 449093': dfs.get('user_access_histories', pd.DataFrame()).query('users_course_id == @EXAMPLE_USERS_COURSE_ID')
}

for name, frame in examples.items():
    if frame is None or frame.empty:
        print(f"{name}: данных нет")
        continue
    print(f"
{name} > shape {frame.shape}")
    display(frame.head())


### 3.8 Гипотезы по оттоку

1. **Долгий перерыв перед окончанием доступа.** `preexpiry_inactivity_flag` + низкий `progress_pct` сигнализируют о высоком риске.
2. **Активность без прогресса.** Большой `total_actions` и низкий `lessons_solved_share` показывают блуждание по курсу.
3. **Курсы без методической поддержки.** Низкие `lessons_with_conspect_share` и `lessons_video_share` усиливают вероятность оттока.
4. **Мало наград и короткие периоды активности.** Отсутствие `has_badge_*` + низкие `active_days` указывают на слабую мотивацию.
5. **Проблемы с заданиями.** Расхождения `solved_tasks_count` vs `wk_solved_task_count` могут отражать технические сбои, после которых студенты уходят.
6. **Учителя как аномальный сегмент.** Фильтрация по `is_teacher` обязательна перед построением target.